In [1]:
import numpy as np
print(np.__version__)


1.26.4


In [8]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("cleaned_tamil.csv")

# ---- CLEAN TEXT ----
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip() != ""]

# ---- LABEL MAP ----
label_map = {"C": 0, "N": 1, "S": 2, "W": 3}
df["label"] = df["label"].map(label_map)

# Remove rows with invalid labels
df = df.dropna(subset=["label"])

# =========================
# TRAIN / VALIDATION SPLIT
# =========================
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

# =========================
# LOAD TAMIL BERT MODEL
# =========================
model_name = "l3cube-pune/tamil-bert"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

# =========================
# METRICS
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), dim=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

# =========================
# TRAINING CONFIG
# =========================
training_args = TrainingArguments(
    output_dir="./dialect_model",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=11,
    weight_decay=0.01,
    logging_dir="./logs"
)


# =========================
# TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# =========================
# TRAIN
# =========================
trainer.train()

trainer.save_model("./final_dialect_model")

print("✅ Training complete. Model saved.")


c:\Users\Student\Desktop\Dialet\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 1049/1049 [00:00<00:00, 30887.45 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/tamil-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Student\Desktop\Dialet\.venv\Lib\site-packages\accelerate\accelerator.py:446: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `a

{'eval_loss': 1.2894833087921143, 'eval_accuracy': 0.34318398474737843, 'eval_f1': 0.175367259771549, 'eval_runtime': 3.5285, 'eval_samples_per_second': 297.294, 'eval_steps_per_second': 18.705, 'epoch': 1.0}


 17%|█▋        | 501/2893 [01:44<08:02,  4.95it/s]

{'loss': 1.2761, 'learning_rate': 1.6543380573798828e-05, 'epoch': 1.9}


 18%|█▊        | 527/2893 [01:52<48:52,  1.24s/it]

...

{'loss': 0.3839, 'learning_rate': 2.716902868994124e-06, 'epoch': 9.51}


 91%|█████████ | 2631/2893 [09:27<05:25,  1.24s/it]

{'eval_loss': 0.8891955614089966, 'eval_accuracy': 0.7197330791229742, 'eval_f1': 0.7218378845431809, 'eval_runtime': 3.5699, 'eval_samples_per_second': 293.847, 'eval_steps_per_second': 18.488, 'epoch': 10.0}


100%|██████████| 2893/2893 [10:23<00:00,  4.64it/s]


{'eval_loss': 0.882706880569458, 'eval_accuracy': 0.7283126787416587, 'eval_f1': 0.7316596378408157, 'eval_runtime': 3.5597, 'eval_samples_per_second': 294.691, 'eval_steps_per_second': 18.541, 'epoch': 11.0}
{'train_runtime': 623.8779, 'train_samples_per_second': 73.93, 'train_steps_per_second': 4.637, 'train_loss': 0.713903472615574, 'epoch': 11.0}
✅ Training complete. Model saved.


In [9]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./final_dialect_model",
    tokenizer="l3cube-pune/tamil-bert"
)


c:\Users\Student\Desktop\Dialet\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [10]:
label_reverse = {0:"C", 1:"N", 2:"S", 3:"W"}

text = "நான் போறேன்"
pred = classifier(text)[0]

dialect = label_reverse[int(pred["label"].split("_")[-1])]
print("Predicted dialect:", dialect)


Predicted dialect: W


In [11]:
texts = [
    "நான் போறேன்",
    "எங்கே போகிறாய்",
    "வாங்கிட்டு வா"
]

for t in texts:
    pred = classifier(t)[0]
    dialect = label_reverse[int(pred["label"].split("_")[-1])]
    print(t, "→", dialect)


நான் போறேன் → W
எங்கே போகிறாய் → W
வாங்கிட்டு வா → W


In [12]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from transformers import Trainer

# Get predictions on validation set
predictions = trainer.predict(val_ds)

logits = predictions.predictions
labels = predictions.label_ids

preds = np.argmax(logits, axis=1)

# Reverse label map
label_names = ["C", "N", "S", "W"]

# =========================
# Accuracy report
# =========================
print("\n=== Classification Report ===")
print(classification_report(labels, preds, target_names=label_names))

# =========================
# Confusion matrix
# =========================
cm = confusion_matrix(labels, preds)

print("\n=== Confusion Matrix ===")
print(cm)


100%|██████████| 66/66 [00:03<00:00, 20.90it/s]


=== Classification Report ===
              precision    recall  f1-score   support

           C       0.89      0.85      0.87       201
           N       0.82      0.69      0.75       360
           S       0.59      0.67      0.63       290
           W       0.67      0.77      0.72       198

    accuracy                           0.73      1049
   macro avg       0.74      0.74      0.74      1049
weighted avg       0.74      0.73      0.73      1049


=== Confusion Matrix ===
[[170   1  11  19]
 [  1 247 100  12]
 [  4  48 195  43]
 [ 15   6  25 152]]
